# Future Fixing: Temporary Reset Pipeline

This notebook is a **temporary experiment** to validate a cleaner formulation without changing the full repo pipeline.

Reset choices implemented:
- predict `log1p(total_delivery_days)` (not margin directly),
- cap training target at 60 days,
- engineer seller historical handling-time feature,
- use strict temporal split with 2018 as test,
- use a single regressor (XGBoost, fallback to HistGradientBoosting).

If this performs better, we can promote pieces to the main notebooks later.

In [6]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, precision_score, recall_score, f1_score

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception:
    HAS_XGB = False
    from sklearn.ensemble import HistGradientBoostingRegressor

SEED = 42
DATA_PATH = Path("../datasets/processed/olist_merged_clean.csv")

In [7]:
df = pd.read_csv(DATA_PATH)

for col in [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# Core target and helper columns
work = df.copy()
work["total_delivery_days"] = (
    work["order_delivered_customer_date"] - work["order_purchase_timestamp"]
).dt.days
work["seller_handling_days"] = (
    work["order_delivered_carrier_date"] - work["order_approved_at"]
).dt.days

work = work.dropna(subset=["order_purchase_timestamp", "total_delivery_days", "estimated_delivery_days"]).copy()
work = work[(work["total_delivery_days"] >= 0) & (work["total_delivery_days"] <= 200)]
work = work[(work["estimated_delivery_days"] >= 0) & (work["estimated_delivery_days"] <= 120)]

work["volumetric_weight"] = (
    work["max_length_cm"] * work["max_width_cm"] * work["max_height_cm"]
) / 6000.0
work["freight_ratio"] = work["total_freight"] / (work["total_price"].abs() + 1.0)

print("Shape after base cleaning:", work.shape)
print("Late rate:", round(work["is_late"].mean(), 4))

Shape after base cleaning: (96183, 38)
Late rate: 0.0794


In [8]:
# Temporal split: 2018 onward as test
cutoff_test = pd.Timestamp("2018-01-01")

trainval = work[work["order_purchase_timestamp"] < cutoff_test].copy()
test = work[work["order_purchase_timestamp"] >= cutoff_test].copy()

# Inside pre-2018 data, use last 20% by time as validation
cutoff_val = trainval["order_purchase_timestamp"].quantile(0.80)
train = trainval[trainval["order_purchase_timestamp"] <= cutoff_val].copy()
val = trainval[trainval["order_purchase_timestamp"] > cutoff_val].copy()

print("Train/Val/Test sizes:", len(train), len(val), len(test))

# Historical seller handling feature (train-history only)
# Uses seller_zip_code_prefix as seller proxy available in merged dataset.
def add_hist_seller_handling(train_df, val_df, test_df):
    prior = train_df["seller_handling_days"].dropna().mean()

    tmp = train_df[["seller_zip_code_prefix", "order_purchase_timestamp", "seller_handling_days"]].copy()
    tmp = tmp.sort_values(["seller_zip_code_prefix", "order_purchase_timestamp"])

    grp = tmp.groupby("seller_zip_code_prefix")["seller_handling_days"]
    # Expanding mean over past observations only
    tmp["seller_hist_handling"] = grp.transform(lambda s: s.expanding().mean().shift(1))

    # Backfill train missing with seller mean then global prior
    seller_mean_map = train_df.groupby("seller_zip_code_prefix")["seller_handling_days"].mean().to_dict()
    train_hist = tmp["seller_hist_handling"].fillna(tmp["seller_zip_code_prefix"].map(seller_mean_map)).fillna(prior)
    train_df = train_df.copy()
    train_df["seller_hist_handling"] = train_hist.values

    for frame in (val_df, test_df):
        frame["seller_hist_handling"] = frame["seller_zip_code_prefix"].map(seller_mean_map).fillna(prior)

    return train_df, val_df.copy(), test_df.copy()

train, val, test = add_hist_seller_handling(train, val, test)

# Smoothed target encoding (train-only) for category and seller proxy
prior_late = train["is_late"].mean()

def smoothed_te_map(series_key, y, m=30):
    stats = pd.DataFrame({"k": series_key, "y": y}).groupby("k")["y"].agg(["mean", "count"]).reset_index()
    stats["te"] = (stats["mean"] * stats["count"] + prior_late * m) / (stats["count"] + m)
    return dict(zip(stats["k"], stats["te"]))

seller_te_map = smoothed_te_map(train["seller_zip_code_prefix"], train["is_late"], m=40)
cat_te_map = smoothed_te_map(train["primary_category"], train["is_late"], m=25)

for frame in (train, val, test):
    frame["seller_te"] = frame["seller_zip_code_prefix"].map(seller_te_map).fillna(prior_late)
    frame["cat_te"] = frame["primary_category"].map(cat_te_map).fillna(prior_late)

# Target transformation + train-only capping
y_train_days_raw = train["total_delivery_days"].copy()
y_val_days = val["total_delivery_days"].copy()
y_test_days = test["total_delivery_days"].copy()

y_train_days_capped = y_train_days_raw.clip(upper=60)
y_train_log = np.log1p(y_train_days_capped)

print("Training target cap applied at 60 days")

Train/Val/Test sizes: 34727 8682 52774
Training target cap applied at 60 days


In [9]:
num_features = [
    "distance_km",
    "max_weight_g",
    "total_freight",
    "estimated_delivery_days",
    "payment_installments",
    "same_state",
    "volumetric_weight",
    "seller_hist_handling",
    "freight_ratio",
    "seller_te",
    "cat_te",
]
cat_features = ["payment_type", "customer_state", "seller_state", "primary_category"]
all_features = num_features + cat_features

X_train = train[all_features].copy()
X_val = val[all_features].copy()
X_test = test[all_features].copy()

pre = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imp", SimpleImputer(strategy="median")),
                    ("sc", StandardScaler(with_mean=False)),
                ]
            ),
            num_features,
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imp", SimpleImputer(strategy="most_frequent")),
                    ("ohe", OneHotEncoder(handle_unknown="ignore", min_frequency=0.01)),
                ]
            ),
            cat_features,
        ),
    ],
    remainder="drop",
)

if HAS_XGB:
    model = XGBRegressor(
        n_estimators=600,
        learning_rate=0.04,
        max_depth=7,
        subsample=0.9,
        colsample_bytree=0.85,
        objective="reg:squarederror",
        random_state=SEED,
        n_jobs=-1,
    )
    model_name = "xgb_regressor"
else:
    model = HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_depth=8,
        max_iter=500,
        random_state=SEED,
    )
    model_name = "hist_gb_fallback"

pipe = Pipeline(steps=[("pre", pre), ("model", model)])
pipe.fit(X_train, y_train_log)

pred_val_days = np.expm1(pipe.predict(X_val)).clip(min=0)
pred_test_days = np.expm1(pipe.predict(X_test)).clip(min=0)

reset_regression_metrics = pd.DataFrame([
    {
        "model": model_name,
        "val_mae": mean_absolute_error(y_val_days, pred_val_days),
        "val_rmse": np.sqrt(mean_squared_error(y_val_days, pred_val_days)),
        "val_r2": r2_score(y_val_days, pred_val_days),
        "test_mae": mean_absolute_error(y_test_days, pred_test_days),
        "test_rmse": np.sqrt(mean_squared_error(y_test_days, pred_test_days)),
        "test_r2": r2_score(y_test_days, pred_test_days),
    }
])

display(reset_regression_metrics.round(4))

,model,val_mae,val_rmse,val_r2,test_mae,test_rmse,test_r2
0,xgb_regressor,6.7893,11.0512,0.0238,4.9769,8.283,0.2228


In [10]:
# Operational proxy: convert predicted total days into late-risk decision
val_late = val["is_late"].astype(int)
test_late = test["is_late"].astype(int)

# A simple monotonic mapper from predicted delay to late probability (without extra model complexity)
pred_val_delay = pred_val_days - val["estimated_delivery_days"].values
pred_test_delay = pred_test_days - test["estimated_delivery_days"].values

from sklearn.isotonic import IsotonicRegression
risk_map = IsotonicRegression(y_min=0.0, y_max=1.0, increasing=True, out_of_bounds="clip")
risk_map.fit(pred_val_delay, val_late.values)
risk_test = risk_map.predict(pred_test_delay)

policy_rows = []
for t in np.linspace(0.05, 0.95, 91):
    pred_alert = (risk_test >= t).astype(int)
    alert_rate = pred_alert.mean()
    if alert_rate > 0.20:
        continue

    tp = int(((pred_alert == 1) & (test_late.values == 1)).sum())
    fp = int(((pred_alert == 1) & (test_late.values == 0)).sum())
    fn = int(((pred_alert == 0) & (test_late.values == 1)).sum())

    policy_rows.append(
        {
            "threshold": float(t),
            "alert_rate": float(alert_rate),
            "precision": precision_score(test_late, pred_alert, zero_division=0),
            "recall": recall_score(test_late, pred_alert, zero_division=0),
            "f1": f1_score(test_late, pred_alert, zero_division=0),
            "business_cost": int(fp * 15 + fn * 155),
        }
    )

reset_policy = pd.DataFrame(policy_rows).sort_values("business_cost")
reset_best_policy = reset_policy.head(1)

display(reset_best_policy.round(4))

,threshold,alert_rate,precision,recall,f1,business_cost
2,0.26,0.1779,0.1464,0.284,0.1932,657115


## Next-Step Upgrade Experiment (Seller Latency + Route Encoding + Dedicated Classifier)

This block tests the requested next steps on top of the reset baseline:
- weighted recent seller handling feature (last 5 orders),
- seller late-momentum feature over the last 14 days,
- route-pair target encoding (`seller_state -> customer_state`),
- dedicated XGBoost classifier,
- optional hurdle stacking feature from regression prediction (`predicted_total_days`).

In [13]:
from sklearn.metrics import average_precision_score, roc_auc_score

try:
    from xgboost import XGBClassifier
    HAS_XGB_CLS = True
except Exception:
    HAS_XGB_CLS = False

# Rebuild full sorted frame for strictly past-only dynamic seller features
wf = work.sort_values("order_purchase_timestamp").copy()

# Define seller handoff lateness threshold from TRAIN only
seller_handoff_late_threshold = train["seller_handling_days"].dropna().quantile(0.75)
if pd.isna(seller_handoff_late_threshold):
    seller_handoff_late_threshold = 3.0

wf["seller_handoff_late_flag"] = (wf["seller_handling_days"] > seller_handoff_late_threshold).astype(int)

# Weighted moving average of seller handling over last 5 orders (past only)
def weighted_last5(series):
    w = np.array([1, 2, 3, 4, 5], dtype=float)
    vals = series.values.astype(float)
    out = np.full_like(vals, np.nan)
    for i in range(len(vals)):
        start = max(0, i - 5)
        hist = vals[start:i]
        if len(hist) == 0:
            continue
        ww = w[-len(hist):]
        out[i] = np.dot(hist, ww) / ww.sum()
    return pd.Series(out, index=series.index)

wf["seller_wma_handling_5"] = (
    wf.groupby("seller_zip_code_prefix", group_keys=False)["seller_handling_days"]
    .apply(weighted_last5)
)

# Seller late momentum in last 14 days (past only)
def seller_late_momentum_14d(frame):
    out = pd.Series(index=frame.index, dtype=float)
    for _, g in frame.groupby("seller_zip_code_prefix"):
        g2 = g[["order_purchase_timestamp", "seller_handoff_late_flag"]].sort_values("order_purchase_timestamp")
        s = g2.set_index("order_purchase_timestamp")["seller_handoff_late_flag"]
        m = s.rolling(window="14D", min_periods=3).mean().shift(1)
        out.loc[g2.index] = m.values
    return out

wf["seller_late_momentum_14d"] = seller_late_momentum_14d(wf)

# Route-pair target encoding from train only
train_idx = train.index
val_idx = val.index
test_idx = test.index

route_key_train = train["seller_state"].astype(str) + "->" + train["customer_state"].astype(str)
prior = train["is_late"].mean()
route_stats = pd.DataFrame({"rk": route_key_train, "y": train["is_late"]}).groupby("rk")["y"].agg(["mean", "count"]).reset_index()
route_stats["route_pair_te"] = (route_stats["mean"] * route_stats["count"] + prior * 30) / (route_stats["count"] + 30)
route_map = dict(zip(route_stats["rk"], route_stats["route_pair_te"]))

wf["route_key"] = wf["seller_state"].astype(str) + "->" + wf["customer_state"].astype(str)
wf["route_pair_te"] = wf["route_key"].map(route_map).fillna(prior)

# Add reset features used earlier so this block is self-contained
seller_hist_map = train.groupby("seller_zip_code_prefix")["seller_handling_days"].mean().to_dict()
wf["seller_hist_handling"] = wf["seller_zip_code_prefix"].map(seller_hist_map)

seller_te_stats = train.groupby("seller_zip_code_prefix")["is_late"].agg(["mean", "count"]).reset_index()
seller_te_stats["seller_te"] = (seller_te_stats["mean"] * seller_te_stats["count"] + prior * 40) / (seller_te_stats["count"] + 40)
seller_te_map_local = dict(zip(seller_te_stats["seller_zip_code_prefix"], seller_te_stats["seller_te"]))
wf["seller_te"] = wf["seller_zip_code_prefix"].map(seller_te_map_local).fillna(prior)

cat_te_stats = train.groupby("primary_category")["is_late"].agg(["mean", "count"]).reset_index()
cat_te_stats["cat_te"] = (cat_te_stats["mean"] * cat_te_stats["count"] + prior * 25) / (cat_te_stats["count"] + 25)
cat_te_map_local = dict(zip(cat_te_stats["primary_category"], cat_te_stats["cat_te"]))
wf["cat_te"] = wf["primary_category"].map(cat_te_map_local).fillna(prior)

# Fill dynamic feature NaNs safely
global_handling = train["seller_handling_days"].dropna().mean()
wf["seller_wma_handling_5"] = wf["seller_wma_handling_5"].fillna(global_handling)
wf["seller_hist_handling"] = wf["seller_hist_handling"].fillna(global_handling)
wf["seller_late_momentum_14d"] = wf["seller_late_momentum_14d"].fillna(train["is_late"].mean())

# Split back
train2 = wf.loc[train_idx].copy()
val2 = wf.loc[val_idx].copy()
test2 = wf.loc[test_idx].copy()

# Dedicated classifier features
num_cls = [
    "distance_km",
    "max_weight_g",
    "total_freight",
    "estimated_delivery_days",
    "payment_installments",
    "same_state",
    "volumetric_weight",
    "seller_hist_handling",
    "seller_wma_handling_5",
    "seller_late_momentum_14d",
    "freight_ratio",
    "seller_te",
    "cat_te",
    "route_pair_te",
]
cat_cls = ["payment_type", "customer_state", "seller_state", "primary_category"]

# Add hurdle stacking feature from regression outputs
train2["pred_total_days_stack"] = np.expm1(pipe.predict(X_train)).clip(min=0)
val2["pred_total_days_stack"] = pred_val_days
test2["pred_total_days_stack"] = pred_test_days

variants = {
    "cls_no_stack": num_cls + cat_cls,
    "cls_with_stack": num_cls + ["pred_total_days_stack"] + cat_cls,
}

def fit_eval_classifier(feature_list, name):
    Xtr = train2[feature_list].copy()
    Xva = val2[feature_list].copy()
    Xte = test2[feature_list].copy()
    ytr = train2["is_late"].astype(int)
    yva = val2["is_late"].astype(int)
    yte = test2["is_late"].astype(int)

    n_num = [c for c in feature_list if c in num_cls or c == "pred_total_days_stack"]
    n_cat = [c for c in feature_list if c in cat_cls]

    pre_cls = ColumnTransformer(
        transformers=[
            ("num", Pipeline(steps=[("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler(with_mean=False))]), n_num),
            ("cat", Pipeline(steps=[("imp", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore", min_frequency=0.01))]), n_cat),
        ],
        remainder="drop",
    )

    if HAS_XGB_CLS:
        pos = max(int(ytr.sum()), 1)
        neg = max(int((1 - ytr).sum()), 1)
        clf = XGBClassifier(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.9,
            colsample_bytree=0.85,
            reg_lambda=1.5,
            objective="binary:logistic",
            eval_metric="aucpr",
            scale_pos_weight=neg / pos,
            random_state=SEED,
            n_jobs=-1,
        )
    else:
        clf = LogisticRegression(max_iter=3000, class_weight="balanced", random_state=SEED)

    model = Pipeline(steps=[("pre", pre_cls), ("clf", clf)])
    model.fit(Xtr, ytr)

    pva = model.predict_proba(Xva)[:, 1]
    pte = model.predict_proba(Xte)[:, 1]

    # Optimize threshold on validation for highest precision with recall >= 20%
    best_t = None
    best_prec = -1
    for t in np.linspace(0.01, 0.99, 99):
        pred_va = (pva >= t).astype(int)
        rec = recall_score(yva, pred_va, zero_division=0)
        prec = precision_score(yva, pred_va, zero_division=0)
        if rec >= 0.20 and prec > best_prec:
            best_prec = prec
            best_t = t

    if best_t is None:
        best_t = 0.5

    pred_te = (pte >= best_t).astype(int)
    topk = int(np.ceil(0.20 * len(pte)))
    topk_idx = np.argsort(-pte)[:topk]
    topk_pred = np.zeros_like(pred_te)
    topk_pred[topk_idx] = 1

    tp = int(((pred_te == 1) & (yte.values == 1)).sum())
    fp = int(((pred_te == 1) & (yte.values == 0)).sum())
    fn = int(((pred_te == 0) & (yte.values == 1)).sum())

    return {
        "variant": name,
        "val_auc": roc_auc_score(yva, pva),
        "val_pr_auc": average_precision_score(yva, pva),
        "test_auc": roc_auc_score(yte, pte),
        "test_pr_auc": average_precision_score(yte, pte),
        "val_selected_threshold": float(best_t),
        "test_precision": precision_score(yte, pred_te, zero_division=0),
        "test_recall": recall_score(yte, pred_te, zero_division=0),
        "test_f1": f1_score(yte, pred_te, zero_division=0),
        "test_business_cost": int(fp * 15 + fn * 155),
        "top20_precision": precision_score(yte, topk_pred, zero_division=0),
        "top20_recall": recall_score(yte, topk_pred, zero_division=0),
    }

rows_cls = [fit_eval_classifier(feats, name) for name, feats in variants.items()]
next_step_classifier_results = pd.DataFrame(rows_cls).sort_values("test_business_cost")
display(next_step_classifier_results.round(4))

,variant,val_auc,val_pr_auc,test_auc,test_pr_auc,val_selected_threshold,test_precision,test_recall,test_f1,test_business_cost,top20_precision,top20_recall
1,cls_with_stack,0.6604,0.2099,0.6321,0.1481,0.17,0.1720,0.2408,0.2006,653465,0.1573,0.3431
0,cls_no_stack,0.6727,0.2318,0.6323,0.1515,0.51,0.1892,0.2061,0.1973,659435,0.1595,0.3481
